In [ ]:
from collections.abc import Callable, Iterable
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from jazz_graph.data.graph_builder.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender, RandomWalkRecommender
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.experiment import BSideExperiment
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run


In [ ]:


def match_recording_traits(recording_traits, artist=None, album=None):
    def starts_with_lower(key, match):
        value = recording_traits[key]
        return value.str.contains(match, flags=re.IGNORECASE)

    if artist and album:
        return recording_traits[starts_with_lower('artist', artist) & starts_with_lower('album', album)]
    elif artist:
        return recording_traits[starts_with_lower('artist', artist)]
    elif album:
        return recording_traits[starts_with_lower('album', album)]


In [ ]:
import psycopg
query = """
            SELECT DISTINCT  -- there are duplicates in recording_to_performer where a musican plays two instuments (Louis)
                recording_to_performer.artist_id,
                recording_to_performer.instrument,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
        """
query = """
            SELECT
                recording_to_performer.*,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
            -- LIMIT 10
        """
with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
    query_result = pd.read_sql(query, conn)
query_result

In [ ]:
query_result[query_result.artist_name.str.startswith('Ella')]

In [ ]:
for row in query_result.instrument.value_counts():
    ...
for key, value in query_result.instrument.value_counts().items():
    print(key, value)

In [ ]:
query_result.album.loc[0]

In [ ]:
for row in query_result.itertuples():
    print(discogs.matching_discog(row))

In [ ]:
from jazz_graph.etl.extract_discogs import MatchDiscogs, InMemDiscogs, is_jazz_album
discogs = MatchDiscogs(InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album))

In [ ]:
discogs.matching_discog((None, None, "Adam's Apple", "Adam's Apple", "Wayne Shorter"))

In [ ]:
embeddings = torch.tensor([
    [.1, .1],
    [1, 0],
    [.1, .2],
    [.2, .1],
    [.3, .3]
])

dp = embeddings @ embeddings[[1, 3]].T
dp
dp.sum(dim=-1)

In [ ]:
INSTRUMENT_CATEGORIES = {
    # Drums
    "drums (drum set)": "drums",
    "electronic drum set": "drums (electronic)",
    "drum machine": "drums (electronic)",

    # Acoustic Piano
    "piano": "piano (acoustic)",
    "grand piano": "piano (acoustic)",
    "upright piano": "piano (acoustic)",
    "prepared piano": "piano (acoustic)",
    "fortepiano": "piano (acoustic)",
    "tack piano": "piano (acoustic)",
    "piano spinet": "piano (acoustic)",

    # Electric Piano / Keys (non-organ)
    "electric piano": "piano (electric)",
    "Rhodes piano": "piano (electric)",
    "Wurlitzer electric piano": "piano (electric)",
    "electric grand piano": "piano (electric)",
    "clavinet": "piano (electric)",
    "celesta": "piano (electric)",
    "harpsichord": "piano (electric)",
    "synthesizer": "piano (electric)",
    "analog synthesizer": "piano (electric)",
    "bass synthesizer": "piano (electric)",
    "string synthesizer": "piano (electric)",
    "keyboard": "piano (electric)",
    "mellotron": "piano (electric)",
    "Moog": "piano (electric)",
    "Minimoog": "piano (electric)",
    "synclavier": "piano (electric)",

    # Organ (distinct jazz tradition)
    "organ": "organ",
    "Hammond organ": "organ",
    "electronic organ": "organ",
    "pipe organ": "organ",
    "theatre organ": "organ",
    "harmonium": "organ",
    "barrel organ": "organ",

    # Double Bass (acoustic)
    "double bass": "bass (acoustic)",
    "acoustic bass guitar": "bass (acoustic)",
    "bass violin": "bass (acoustic)",
    "bass viol": "bass (acoustic)",
    "contrabass saxophone": "bass (acoustic)",  # functional bass role

    # Electric Bass
    "electric bass guitar": "bass (electric)",
    "bass guitar": "bass (electric)",
    "fretless bass": "bass (electric)",
    "electric bass guitar": "bass (electric)",
    "electric upright bass": "bass (electric)",
    "bass pedals": "bass (electric)",
    "keyboard bass": "bass (electric)",
    "washtub bass": "bass (electric)",

    # Acoustic Guitar
    "guitar": "guitar (acoustic)",
    "acoustic guitar": "guitar (acoustic)",
    "classical guitar": "guitar (acoustic)",
    "steel-string acoustic guitar": "guitar (acoustic)",
    "archtop guitar": "guitar (acoustic)",
    "resonator guitar": "guitar (acoustic)",
    "12 string guitar": "guitar (acoustic)",
    "acoustic fretless guitar": "guitar (acoustic)",

    # Electric Guitar
    "electric guitar": "guitar (electric)",
    "guitar synthesizer": "guitar (electric)",
    "pedal steel guitar": "guitar (electric)",
    "lap steel guitar": "guitar (electric)",
    "steel guitar": "guitar (electric)",
    "baritone guitar": "guitar (electric)",
    "electric fretless guitar": "guitar (electric)",
    "ebow": "guitar (electric)",
    "Chapman stick": "guitar (electric)",
    "slide guitar": "guitar (electric)",

    # Trumpet family (keep together — flugelhorn/cornet are same acoustic tradition)
    "trumpet": "trumpet",
    "flugelhorn": "trumpet",
    "cornet": "trumpet",
    "pocket trumpet": "trumpet",
    "piccolo trumpet": "trumpet",
    "bass trumpet": "trumpet",
    "bugle": "trumpet",

    # Trombone
    "trombone": "trombone",
    "bass trombone": "trombone",
    "valve trombone": "trombone",
    "tenor trombone": "trombone",

    # Saxophones — keep all voices distinct
    "soprano saxophone": "saxophone (soprano)",
    "sopranino saxophone": "saxophone (soprano)",
    "alto saxophone": "saxophone (alto)",
    "tenor saxophone": "saxophone (tenor)",
    "baritone saxophone": "saxophone (baritone)",
    "bass saxophone": "saxophone (baritone)",  # close enough
    "saxophone": "saxophone (tenor)",  # generic, tenor is the default assumption in jazz

    # Clarinet
    "clarinet": "clarinet",
    "bass clarinet": "clarinet",
    "alto clarinet": "clarinet",
    "contrabass clarinet": "clarinet",
    "basset clarinet": "clarinet",

    # Flute
    "flute": "flute",
    "alto flute": "flute",
    "bass flute": "flute",
    "piccolo": "flute",
    "recorder": "flute",

    # Vibraphone (signature jazz melodic percussion)
    "vibraphone": "vibraphone",
    "marimba": "vibraphone",  # close enough functionally

    # Violin
    "violin": "violin",
    "electric violin": "violin",
    "fiddle": "violin",
    "viola": "violin",
    "cello": "violin",
    "electric cello": "violin",
    "string quartet": "violin",
    "strings": "violin",
    "bass violin": "violin",
    "tenor violin": "violin",

    # Vocals
    "lead vocals": "vocals (lead)",
    "background vocals": "vocals (other)",
    "choir vocals": "vocals (other)",
    "spoken vocals": "vocals (other)",
    "tenor vocals": "vocals (other)",
    "soprano vocals": "vocals (other)",
    "baritone vocals": "vocals (other)",
    "alto vocals": "vocals (other)",
    "bass vocals": "vocals (other)",
    "vocal": "vocals (other)",
    "other vocals": "vocals (other)",
    "mezzo-soprano vocals": "vocals (other)",
    "countertenor vocals": "vocals (other)",
    "whistling": "vocals (other)",

    # Other brass
    "French horn": "other brass",
    "tuba": "other brass",
    "sousaphone": "other brass",
    "mellophone": "other brass",
    "euphonium": "other brass",
    "baritone horn": "other brass",
    "horn": "other brass",
    "tenor horn / alto horn": "other brass",
    "brass": "other brass",
    "valved brass instruments": "other brass",
    "cornett": "other brass",
    "sackbut": "other brass",

    # Other wind
    "bassoon": "other wind",
    "contrabassoon": "other wind",
    "oboe": "other wind",
    "cor anglais": "other wind",
    "woodwind": "other wind",
    "reeds": "other wind",
    "wind instruments": "other wind",
    "double reed": "other wind",
    "harmonica": "other wind",
    "melodica": "other wind",
    "accordion": "other wind",
    "bandoneón": "other wind",
    "bagpipe": "other wind",
    "concertina": "other wind",

    # Percussion (not drum set)
    "percussion": "percussion",
    "membranophone": "percussion",
    "congas": "percussion",
    "bongos": "percussion",
    "timbales": "percussion",
    "snare drum": "percussion",
    "bass drum": "percussion",
    "tambourine": "percussion",
    "cowbell": "percussion",
    "djembe": "percussion",
    "tabla": "percussion",
    "shekere": "percussion",
    "talking drum": "percussion",
    "gong": "percussion",
    "maracas": "percussion",
    "triangle": "percussion",
    "timpani": "percussion",
    "tubular bells": "percussion",
    "xylophone": "percussion",
    "glockenspiel": "percussion",
    "bell": "percussion",
    "cymbal": "percussion",
    "chimes": "percussion",
    "handclaps": "percussion",
    "shakers": "percussion",
    "cimbalom": "percussion",
    "vibraslap": "percussion",
    "finger snaps": "percussion",
    "foot stomps": "percussion",
    "castanets": "percussion",
    "washboard": "percussion",
    "steelpan": "percussion",
    "marimba": "percussion",
    "finger cymbals": "percussion",
    "hi-hat": "percussion",
    "tom-tom": "percussion",
    "frame drum": "percussion",
    "slit drum": "percussion",
    "Batá drum": "percussion",
    "dunun": "percussion",
    "bendir": "percussion",
    "surdo": "percussion",
    "timbales": "percussion",
    "claves": "percussion",
    "cabasa": "percussion",
    "bongos": "percussion",
    "cuíca": "percussion",
    "berimbau": "percussion",
    "wood block": "percussion",
    "gankogui": "percussion",
    "agogô": "percussion",
    "handbell": "percussion",
    "crotales": "percussion",
    "ganzá": "percussion",
    "ocean drum": "percussion",
    "rainstick": "percussion",
    "rhythm sticks": "percussion",
    "bones": "percussion",
    "darbuka": "percussion",
    "goblet drum": "percussion",
    "sabar": "percussion",
    "zabumba": "percussion",
    "ashiko": "percussion",
    "quinto": "percussion",
    "dohol": "percussion",
    "junjung": "percussion",
    "water drum": "percussion",
    "friction drum": "percussion",
    "mridangam": "percussion",
    "kanjira": "percussion",
    "ghatam": "percussion",

    # Electronic / effects
    "effects": "electronic/effects",
    "sampler": "electronic/effects",
    "vocoder": "electronic/effects",
    "turntable": "electronic/effects",
    "tape": "electronic/effects",
    "EWI": "electronic/effects",
    "Lyricon": "electronic/effects",
    "wind synthesizer": "electronic/effects",
    "talkbox": "electronic/effects",
    "voice synthesizer": "electronic/effects",
    "theremin": "electronic/effects",

    # World / other instruments (too sparse or too niche to signal anything)
    "banjo": "other/world",
    "tenor banjo": "other/world",
    "mandolin": "other/world",
    "ukulele": "other/world",
    "harp": "other/world",
    "electric harp": "other/world",
    "harp guitar": "other/world",
    "oud": "other/world",
    "sitar": "other/world",
    "koto": "other/world",
    "lute": "other/world",
    "zither": "other/world",
    "kora": "other/world",
    "bouzouki": "other/world",
    "charango": "other/world",
    "banjo": "other/world",
    "mbira": "other/world",
    "shakuhachi": "other/world",
    "didgeridoo": "other/world",
    "kazoo": "other/world",
    "whistle": "other/world",
    "tin whistle": "other/world",
    "autoharp": "other/world",
    "guitar family": "other/world",
    "violin family": "other/world",
    "trumpet family": "other/world",
    "bowed string instruments": "other/world",
    "slide brass instruments": "other/world",
    "lamellaphone": "other/world",
    "idiophone": "other/world",
    "shruti box": "other/world",
    "tambura": "other/world",
    "harmonium": "other/world",
    "accordion": "other/world",
    "mandola": "other/world",
    "mandocello": "other/world",
    "cavaquinho": "other/world",
    "fiddle": "other/world",
    "rebab": "other/world",
    "hardingfele": "other/world",
    "bandura": "other/world",
    "Chapman stick": "other/world",
    "guzheng": "other/world",
    "pipa": "other/world",
    "khamak": "other/world",
    "sarod": "other/world",
    "santoor": "other/world",
    "kamancheh": "other/world",
    "setar": "other/world",
    "tar": "other/world",
    "sitar": "other/world",
    "tabla": "other/world",
    "tanpura": "other/world",
    "valiha": "other/world",
    "ngɔni": "other/world",
    "xalam": "other/world",
    "gumbri": "other/world",
    "balafon": "other/world",
    "musical saw": "other/world",
    "waterphone": "other/world",
    "singing bowl": "other/world",
    "glass harp": "other/world",
    "toy piano": "other/world",
    "clavichord": "other/world",
    "harpsichord": "other/world",
    "psaltery": "other/world",
    "dulcimer": "other/world",
    "hammered dulcimer": "other/world",
    "omnichord": "other/world",
    "marímbula": "other/world",
    "taragot": "other/world",
    "duduk": "other/world",
    "suona": "other/world",
    "shehnai": "other/world",
    "kaval": "other/world",
    "ney": "other/world",
    "bansuri": "other/world",
    "quena": "other/world",
    "pan flute": "other/world",
    "shakuhachi": "other/world",
    "transverse flute": "other/world",
    "alto flute": "other/world",
    "sheng": "other/world",
    "fife": "other/world",
    "farfisa": "other/world",
    "tubax": "other/world",
    "tiple": "other/world",
    "charango": "other/world",
    "cuatro": "other/world",
    "berimbau": "other/world",
    "udu": "other/world",
    "caxixi": "other/world",
    "güiro": "other/world",
    "shofar": "other/world",

    # Other personnel (not instruments)
    "guest": "other personnel",
    "assistant": "other personnel",
    "executive": "other personnel",
    "associate": "other personnel",
    "additional": "other personnel",
    "co": "other personnel",
    "task": "other personnel",
    "solo": "other personnel",
    "other instruments": "other personnel",
}

pd.Series(INSTRUMENT_CATEGORIES).value_counts()

In [ ]:
from torch_geometric.nn import GATv2Conv, GATConv, SAGEConv

x_dict = torch.tensor([
    [.1, .2, .3, .4],
    [.1, -.3, .5, .6]
])
edge_index_dict = torch.tensor([
    [1, 0], [0, 1]
])
edge_attrs = torch.tensor([
    [-1., .1, -.8, .2],
    [-1., .1, .8, .2]
])

conv = GATv2Conv(4, 6, edge_dim=4)
conv(x_dict, edge_index_dict, edge_attrs)

In [ ]:
from torch import nn
from collections.abc import Sequence
from torch_geometric.nn import HeteroConv


class GNNModel(nn.Module):
    def __init__(self, hidden_dim, input_dim, metadata, edge_dim: dict[tuple[str, str, str], int], model_type: str | Sequence[str] = 'gatv2', num_layers=3, dropout=0.):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)

        # Create layers dynamically
        self.convs = nn.ModuleList()
        edge_types = metadata[1]
        if isinstance(model_type, str):
            model_type = [model_type for _ in range(len(edge_types))]
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            self.convs.append(
                HeteroConv({
                    key: self._model_type(model_type[i], in_dim, hidden_dim, edge_dim.get(key)) for key in edge_types
                })
            )

    def _model_type(self, model_type, in_dim, hidden_dim, edge_dim):
        if model_type == 'sage':
            if edge_dim is not None:
                raise ValueError("SAGEConv is not supported with edge attributes.")
            return SAGEConv(in_dim, hidden_dim, normalize=True)
        elif model_type == 'gat':
            return GATConv(in_dim, hidden_dim, edge_dim=edge_dim, add_self_loops=False)
        elif model_type == 'gatv2':
            return GATv2Conv(in_dim, hidden_dim, edge_dim=edge_dim, add_self_loops=False)
        else:
            raise ValueError("Expected model type 'sage', 'gat' or 'gatv2'.")

    def forward(self, x_dict, edge_dict, edge_attrs):
        for i, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_dict, edge_attrs)

            # Apply ReLU + dropout to all layers except the last
            if i < self.num_layers - 1:
                x_dict = {key: F.normalize(self.dropout(F.relu(val)), dim=-1) for key, val in x_dict.items()}
            else:
                # Last layer: dropout only (no ReLU)
                x_dict = {key: F.normalize(self.dropout(val), dim=-1) for key, val in x_dict.items()}

        return x_dict

In [ ]:
class JazzModel(nn.Module):
    def __init__(
        self,
        num_performances: int,
        num_artists: int,
        num_songs: int,
        embed_dim: int,
        hidden_dim: int,
        metadata: tuple,
        edge_dim: dict,
        dropout: float = 0.0,
        num_layers=3,
        model_type='sage'
    ):
        super().__init__()

        self.performance_embed = nn.Embedding(num_performances, embed_dim)
        self.song_embed = nn.Embedding(num_songs, embed_dim)
        self.artist_embed = nn.Embedding(num_artists, embed_dim)

        self.gnn = GNNModel(hidden_dim, embed_dim, metadata, edge_dim, model_type, num_layers, dropout)

        self.style_projections = nn.Linear(num_labels, 16)
        self.instrument_embeddings = nn.Embedding(num_instruments, embed_dim)

    def forward(self, x_dict, edge_dict, batch: HeteroData) -> torch.Tensor:
        if hasattr(batch['performance'], 'n_id'):
            x_dict = {
                'performance': self.performance_embed(batch['performance'].n_id),
                'artist': self.artist_embed(batch['artist'].n_id),
                'song': self.song_embed(batch['song'].n_id)
            }
        else:
            x_dict = {
                'performance': self.performance_embed(torch.arange(batch['performance'].num_nodes)),
                'artist': self.artist_embed(torch.arange(batch['artist'].num_nodes)),
                'song': self.song_embed(torch.arange(batch['song'].num_nodes))
            }
        for node in batch.node_types:
            x_dict[node] = torch.concat([batch[node].x, x_dict[node]])
        # batch['song'].x
        # batch['artist'].x
        edge_instruments = batch['artist', 'performs', 'performance'].edge_attrs
        instrument_embeddings = self.instrument_embeddings(edge_instruments)


        x_dict = self.gnn(x_dict, edge_dict, {('artist', 'performs', 'performance'): instrument_embeddings})
        return x_dict
